# Project Midas — full M34 pipeline (Phases I–IV)

End-to-end reproduction of the **Project Midas** revival: legacy B−V photometry → Gaia cross-match → Q-value validation → binary synthesis → Rubin white-dwarf check.

**Prerequisites**

```bash
cd research
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt -r requirements-dev.txt
jupyter lab   # or: jupyter notebook
```

Place legacy CSVs in `data/raw/` (or keep a sibling `Midas/` folder — see `midas.paths`):

| File | Purpose |
|------|--------|
| `Midas Raw Data.csv` | BVR(I) photometry |
| `Members.csv` | Jones & Prosser membership |
| `ISO.csv` | Yonsei–Yale isochrones |

Network stages (Gaia TAP, VizieR catalogs, IR cones) are **optional** when cached files already exist in `data/processed/`.

CLI equivalent: `python scripts/run_reproduction.py --stage all` — see [`REPRODUCTION.md`](../REPRODUCTION.md).

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve research/ whether the kernel cwd is notebooks/ or research/
ROOT = Path.cwd().resolve()
if not (ROOT / "midas").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "midas").is_dir():
    raise RuntimeError("Run from research/ or research/notebooks/")

sys.path.insert(0, str(ROOT))
SCRIPTS = ROOT / "scripts"
PROCESSED = ROOT / "data" / "processed"

from IPython.display import display

from midas.excel import classify_all
from midas.join_table import JOIN_CSV, load_join_table
from midas.paths import PROCESSED as MIDAS_PROCESSED, iso_csv, members_csv, midas_photometry
from midas.pipeline import MidasPipeline, count_accepted
from midas.reddening import DEFAULT_EBV
from midas.synthesis import print_synthesis_report, run_synthesis
from midas.validation import print_confusion_report, run_all_validations
from midas.white_dwarfs import print_wd_report, run_wd_check

EBV = DEFAULT_EBV  # 0.07
TARGET_SINGLES, TARGET_BINARIES = 187, 171

# Set True to query Gaia / VizieR / IR archives (slow; needs network)
FETCH_NETWORK = False

# Matplotlib style (fallback if seaborn theme unavailable)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass
print(f"Research root: {ROOT}")
print(f"E(B−V) = {EBV}")

## 0 — Check raw inputs

In [ ]:
def run_script(name: str, *args: str) -> None:
    path = SCRIPTS / name
    print(f"\n=== {name} {' '.join(args)} ===")
    subprocess.run([sys.executable, str(path), *args], check=True, cwd=ROOT)


raw_checks = {
    "Midas photometry": midas_photometry(),
    "J&P members": members_csv(),
    "YY isochrones": iso_csv(),
}
for label, path in raw_checks.items():
    print(f"{label:20s} → {path} ({path.stat().st_size // 1024} KiB)")

## Phase I — Legacy Midas pipeline

Recompute Mv, Q-values, and Jones–Prosser mating. Verify Excel Control counts (187 singles / 171 binaries).

In [ ]:
_, excel_singles, excel_binaries = classify_all()
pipe = MidasPipeline(ebv=EBV)
pipe.write_csv(PROCESSED / "midas_pipeline.csv")
py_accept = count_accepted(pipe.stars)

phase1 = pd.DataFrame(
    {
        "metric": ["Excel singles", "Excel binaries", "Python accepted (legacy cut)", "Stars in pipeline"],
        "value": [excel_singles, excel_binaries, py_accept, len(pipe.stars)],
        "target": [TARGET_SINGLES, TARGET_BINARIES, None, None],
    }
)
phase1["match"] = phase1.apply(
    lambda r: r["value"] == r["target"] if pd.notna(r["target"]) else None, axis=1
)
display(phase1)

assert excel_singles == TARGET_SINGLES and excel_binaries == TARGET_BINARIES, "Excel regression failed"

In [ ]:
df_pipe = pd.read_csv(PROCESSED / "midas_pipeline.csv")
fig, ax = plt.subplots(figsize=(7, 5))
sample = df_pipe.sample(min(1200, len(df_pipe)), random_state=0)
ax.scatter(sample["bv"], sample["mv"], s=6, alpha=0.35, c="#4a6fa5", label="all stars")
q_bin = (sample["Q"] > 0) & (sample["Q"] <= 1)
ax.scatter(sample.loc[q_bin, "bv"], sample.loc[q_bin, "mv"], s=10, alpha=0.7, c="#c44e52", label="Q ∈ (0, 1]")
ax.invert_yaxis()
ax.set_xlabel("B − V")
ax.set_ylabel("Mv")
ax.set_title(f"Phase I — Midas CMD (E(B−V)={EBV})")
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()

## Phase II — Gaia + published catalogs → join table

Cross-match Midas to Gaia DR3, Cantat-Gaudin, Malofeeva, WOCS, and Jones–Prosser. One row per Midas star with dereddened `bv0`, `mv0`.

In [ ]:
gaia_csv = PROCESSED / "gaia_m34.csv"
if FETCH_NETWORK or not gaia_csv.exists():
    run_script("gaia_cone.py", "--radius-deg", "0.5", "--out", "data/processed/gaia_m34.csv")
else:
    print(f"Using cached Gaia cone: {gaia_csv}")

if FETCH_NETWORK:
    run_script("fetch_published_catalogs.py")
else:
    print("Using cached VizieR tables (cantat_gaudin, malofeeva, wocs_meibom)")

run_script("cross_match.py", "--ebv", str(EBV))

join = load_join_table()
cg = join[join["cg_member"] == True]  # noqa: E712
phase2 = pd.Series(
    {
        "Midas rows": len(join),
        "Gaia matched": join["gaia_source_id"].notna().sum(),
        "CG members (P≥0.7)": len(cg),
        "Malofeeva flags": join["malofeeva"].sum(),
        "WOCS overlap": join["wocs"].sum(),
    },
    name="Phase II",
)
display(phase2.to_frame("count"))

## Phase III — Validation (Q vs external truth)

Compare legacy Q-value cuts to Malofeeva IR flags, WOCS RV variability, and Gaia RUWE on Cantat-Gaudin members.

In [ ]:
validation = run_all_validations(ebv=EBV, refresh_pipeline=True, members_only=True)

for key in ("malofeeva", "wocs", "ruwe"):
    print_confusion_report(validation[key])

val_rows = []
for key, label in (("malofeeva", "Malofeeva IR"), ("wocs", "WOCS PRV"), ("ruwe", "Gaia RUWE")):
    m = validation[key]
    val_rows.append(
        {
            "truth set": label,
            "N": m["n"],
            "precision": m["precision"],
            "recall": m["recall"],
            "F1": m["f1"],
        }
    )
display(pd.DataFrame(val_rows).round(3))

In [ ]:
roc = validation["roc_malofeeva"]["curve"]
fpr = [p["fpr"] for p in roc]
tpr = [p["tpr"] for p in roc]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].plot(fpr, tpr, color="#4c72b0", lw=2)
axes[0].plot([0, 1], [0, 1], "--", color="#999", lw=1)
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC — Q vs Malofeeva (CG members)")

bins = validation["completeness_malofeeva"]["bins"]
mv_mid = [(b["mv_lo"] + b["mv_hi"]) / 2 for b in bins if b["recall"] is not None]
recall = [b["recall"] for b in bins if b["recall"] is not None]
ci_lo = [b["recall_ci_lo"] for b in bins if b["recall"] is not None]
ci_hi = [b["recall_ci_hi"] for b in bins if b["recall"] is not None]
axes[1].errorbar(mv_mid, recall, yerr=[np.array(recall) - np.array(ci_lo), np.array(ci_hi) - np.array(recall)], fmt="o-", capsize=3)
axes[1].set_xlabel("Mv")
axes[1].set_ylabel("Recall vs Malofeeva")
axes[1].set_title("Completeness by magnitude (bootstrap 95% CI)")
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

## Phase IV — Synthesis & method comparison

Deduplicated binary fraction across channels (Q, Malofeeva, Excel, WOCS PRV, RUWE) and mass-binned union fractions.

In [ ]:
synthesis = run_synthesis(ebv=EBV, refresh_pipeline=False, members_only=True)
print_synthesis_report(synthesis)

channels = pd.DataFrame(
    [{"channel": k, "count": v} for k, v in synthesis["overall"]["channel_counts"].items() if k != "union"]
)
overlap_keys = pd.Series(synthesis["overlap"]["key_sets"], name="count")
display(channels)
display(overlap_keys.to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ch_items = [(k, v) for k, v in synthesis["overall"]["channel_counts"].items() if k != "union"]
labels = [k for k, _ in ch_items]
counts = [v for _, v in ch_items]
axes[0].barh(labels, counts, color="#55a868")
axes[0].set_xlabel("Stars flagged")
axes[0].set_title(f"Channel hits (N={synthesis['overall']['n']} CG members)")
axes[0].invert_yaxis()

union_bins = synthesis["by_mass"]["union"]
mass_mid = [(b["mass_lo"] + b["mass_hi"]) / 2 for b in union_bins]
frac = [b["fraction"] for b in union_bins]
axes[1].plot(mass_mid, frac, "o-", color="#c44e52")
axes[1].set_xlabel("Mass (M☉, YY 0.2 Gyr)")
axes[1].set_ylabel("Union binary fraction")
axes[1].set_title("Union fraction vs mass (Malofeeva-dominated)")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
ir_join = PROCESSED / "m34_join_ir.csv"
if FETCH_NETWORK or not ir_join.exists():
    if FETCH_NETWORK or not (PROCESSED / "twomass_m34.csv").exists():
        run_script("fetch_ir_photometry.py", "--verify")
    run_script("merge_ir_photometry.py")

if ir_join.exists():
    ir = pd.read_csv(ir_join)
    cg_ir = ir[(ir["cg_member"] == True) & ir["w2_bp"].notna()]  # noqa: E712
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(cg_ir["bv0"], cg_ir["w2_bp"], s=8, alpha=0.25, c="#888", label="CG members")
    mal = cg_ir["malofeeva"] == True  # noqa: E712
    ax.scatter(cg_ir.loc[mal, "bv0"], cg_ir.loc[mal, "w2_bp"], s=12, alpha=0.6, c="#4c72b0", label="Malofeeva")
    ax.set_xlabel("de-reddened B − V (bv0)")
    ax.set_ylabel("W2 − BP (mag)")
    ax.set_title("Method comparison — IR pseudocolor vs B−V")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
else:
    print("Skip W2−BP plot — run merge_ir_photometry.py or set FETCH_NETWORK=True")

## White dwarf check (Rubin LAWDS)

Cross-match 44 Rubin et al. (2008) candidates to Gaia DR3 and assess cluster astrometry.

In [ ]:
if FETCH_NETWORK or not (ROOT / "data" / "raw" / "rubin_lawds_m34.csv").exists():
    run_script("fetch_rubin_wd.py")

wd = run_wd_check()
print_wd_report(wd)

wd_df = pd.DataFrame(wd["candidates"])[["lawds_id", "v_mag", "paper_cluster_member", "astrometry_verdict", "gaia_source_id"]]
display(wd_df.head(12))

## Summary

Key numbers from this run. Full write-up: [midasastronomy.com/findings](https://midasastronomy.com/findings).

In [ ]:
summary = pd.DataFrame(
    [
        ("Excel singles / binaries", f"{excel_singles} / {excel_binaries}"),
        ("Midas pipeline stars", str(len(pipe.stars))),
        ("Join table rows", str(len(join))),
        ("CG members", str(len(cg))),
        ("Q vs Malofeeva F1", f"{validation['malofeeva']['f1']:.3f}"),
        ("Union binary fraction", f"{synthesis['overall']['fraction_union']:.1%}"),
        ("LAWDS Gaia matched", str(wd["summary"]["n_gaia_matched"])),
        ("Paper cluster DAs", str(wd["summary"]["n_paper_cluster_members"])),
    ],
    columns=["Finding", "Value"],
)
display(summary)

# Optional: refresh website JSON (from research/)
EXPORT_WEB = False
if EXPORT_WEB:
    run_script("build_web_all.py")
    print("Web JSON updated under ../web/src/data/")